# 01 - Seed data profile

**Soro Nua Campaign Predictor - preparation of phase 7 (D-044).**
What the starting dataset (8,000 rows, `data/seed/`) really contains, before any model is trained.
Nothing is written to the database: this notebook only reads the CSV and writes figures to `_out/`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import common as c

c.OUT_DIR.mkdir(exist_ok=True)
(c.OUT_DIR / "figures").mkdir(exist_ok=True)
pd.set_option("display.precision", 4)
raw = pd.read_csv(c.SEED_CSV_PATH)
df = c.load_seed()
assert raw.shape == (8000, 20), raw.shape
assert raw.isna().sum().sum() == 0
print("rows, columns:", raw.shape, "| missing values:", int(raw.isna().sum().sum()))

rows, columns: (8000, 20) | missing values: 0


## 1. Columns and ranges

In [2]:
summary = raw.describe().T[["min", "max", "mean", "std"]]
summary

,min,max,mean,std
CustomerID,8000.0000,15999.0000,11999.5000,2309.5454
Age,18.0000,69.0000,43.6255,14.9028
Income,20014.0000,149986.0000,84664.1968,37580.3879
AdSpend,100.0548,9997.9148,5000.9448,2838.0382
ClickThroughRate,0.0100,0.3000,0.1548,0.0840
ConversionRate,0.0100,0.2000,0.1044,0.0549
WebsiteVisits,0.0000,49.0000,24.7516,14.3123
PagesPerVisit,1.0004,9.9991,5.5493,2.6074
TimeOnSite,0.5017,14.9953,7.7277,4.2282
SocialShares,0.0000,99.0000,49.7998,28.9012


## 2. Categories, constant columns and the ignored `Conversion` column
`AdvertisingPlatform` and `AdvertisingTool` have a single value, so the training rule (LLD 5.2) leaves them out.
`Conversion` is an individual outcome, ignored by the system. `CustomerID` is an identifier, never a model input.

In [3]:
for column in ["Gender", "CampaignChannel", "CampaignType", "AdvertisingPlatform", "AdvertisingTool", "Conversion"]:
    print(column, raw[column].value_counts().to_dict())
assert raw["CustomerID"].is_unique and raw["CustomerID"].min() == 8000 and raw["CustomerID"].max() == 15999
constants = [col for col in raw.columns if raw[col].nunique() == 1]
print("\nconstant columns:", constants)
assert constants == ["AdvertisingPlatform", "AdvertisingTool"]
print("share of Conversion = 1:", round(raw["Conversion"].mean(), 4))

Gender {'Female': 4839, 'Male': 3161}
CampaignChannel {'Referral': 1719, 'PPC': 1655, 'Email': 1557, 'SEO': 1550, 'Social Media': 1519}
CampaignType {'Conversion': 2077, 'Awareness': 1988, 'Consideration': 1988, 'Retention': 1947}
AdvertisingPlatform {'IsConfid': 8000}
AdvertisingTool {'ToolConfid': 8000}
Conversion {1: 7012, 0: 988}

constant columns: ['AdvertisingPlatform', 'AdvertisingTool']
share of Conversion = 1: 0.8765


## 3. The three targets
Click-through rate and conversion rate look uniform and website visits is a flat count: typical of fabricated data.

In [4]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, col in zip(axes, ["ClickThroughRate", "ConversionRate", "WebsiteVisits", "AdSpend"]):
    ax.hist(raw[col], bins=30, color="#5b7a99")
    ax.set_title(col)
fig.tight_layout()
fig.savefig(c.OUT_DIR / "figures" / "01_distributions.png", dpi=110)
plt.show()

## 4. Do the inputs carry any signal?
Pearson correlation between every numeric input and the three targets.

In [5]:
numeric_inputs = list(c.NUMERIC)
corr = df[numeric_inputs + list(c.TARGETS)].corr().loc[numeric_inputs, list(c.TARGETS)]
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(corr, cmap="RdBu_r", vmin=-0.2, vmax=0.2)
ax.set_xticks(range(3), c.TARGETS, rotation=30, ha="right")
ax.set_yticks(range(len(numeric_inputs)), numeric_inputs)
fig.colorbar(image, label="correlation")
fig.tight_layout()
fig.savefig(c.OUT_DIR / "figures" / "01_correlations.png", dpi=110)
plt.show()
max_corr = float(corr.abs().to_numpy().max())
print("largest absolute correlation between an input and a target:", round(max_corr, 4))
assert max_corr < 0.05

largest absolute correlation between an input and a target: 0.0232


## 5. Target averages by channel and by type
If channel or type mattered, these averages would differ clearly. They do not.

In [6]:
by_channel = df.groupby("campaign_channel")[list(c.TARGETS)].mean()
by_type = df.groupby("campaign_type")[list(c.TARGETS)].mean()
display(by_channel)
display(by_type)
relative_spread = float(((by_channel.max() - by_channel.min()) / by_channel.mean()).max())
print("largest relative spread across channels:", round(relative_spread, 3))

,click_through_rate,conversion_rate,website_visits
campaign_channel,,,
Email,0.1556,0.1049,24.8786
PPC,0.1583,0.1041,24.1553
Referral,0.1517,0.1031,24.8906
SEO,0.1532,0.1035,24.7400
Social Media,0.1555,0.1066,25.1257


,click_through_rate,conversion_rate,website_visits
campaign_type,,,
Awareness,0.1558,0.1044,24.7596
Consideration,0.1518,0.1051,25.3023
Conversion,0.1557,0.1049,24.3760
Retention,0.1561,0.1032,24.5819


largest relative spread across channels: 0.043


## What this means
* The dataset is fabricated (no real campaign, no personal data): inputs and targets are close to independent.
* Two columns are constant and one is an identifier: 13 inputs remain (10 numeric + channel, type, gender), i.e. 21 model columns.
* With such data no model can be reliable: notebook 02 trains the three models and measures it honestly.